# Battle RL — 通用远程 PPO GPU Worker

连接到本地 hub 的 cloudflared tunnel，以 **pull** 或 **push** 模式领取任意 PPO 任务。

## 两种模式

| 模式 | 说明 | 适用场景 |
|---|---|---|
| **pull**（默认） | Worker 轮询 hub 领取 job（hub 侧 cloudflared） | 标准的 Kaggle/Colab 接入方式 |
| **push** | Worker 启动服务端 + 本地 cloudflared，hub 主动推送 job | hub 侧不方便跑 tunnel 时 |

## 使用前

1. 确保 hub 端已启动：`bun tools/hub-start.ts <course>`（pull 模式）或 hub 已配置 push 节点
2. 在下方的 `⚙️ 参数配置` 单元格填入连接信息
3. 依次运行各单元格

> **通用性**：本 notebook 不绑定任何特定课程——hub 发布的 job manifest 携带完整课程上下文，
> worker 按 manifest 中的 reward 公式和超参执行 PPO。

---
## ⚙️ 参数配置

In [ ]:
# @title 填入连接参数
import os
import sys
import time

# ── 模式选择 ──
MODE = "pull"            # "pull" 或 "push"

# ── Pull 模式参数（MODE="pull"） ──
# HUB_URL 和 HUB_TOKEN 由 hub 端提供：
#   - 运行 hub-start.ts 后，终端会打印 "📋 Kaggle 粘贴用" 的 URL 和 token
#   - 或从 rl-config.json 的 rl.remote_hub_url / rl.remote_token 获取
HUB_URL = "https://your-tunnel.trycloudflare.com"
HUB_TOKEN = "YOUR_TOKEN_HERE"

# ── Push 模式参数（MODE="push"） ──
# 本机运行 cloudflared 暴露端口，hub 侧 push job 到此节点
PUSH_PORT = 8790
PUSH_TOKEN = "YOUR_TOKEN_HERE"
# cloudflared 路径（通常自动在 PATH 中，可留空）
CLOUDFLARED_PATH = ""
# push 模式 bootstrap 代码来源：服务进程要先能 import remote_worker_serve，而干净
# Kaggle/Colab 运行时里没有仓库代码。留空 = 拉 GitHub main tarball。
# （job 真正执行用的是 hub 随 job 下发的 code.zip；bootstrap 只负责把服务拉起来。）
PUSH_CODE_URL = ""

# ── 通用参数 ──
# 保活时长：Kaggle GPU 最长 9h，Colab 免费版约 90min 空闲回收。
# 短腿按需调小（如 c6b-margin max_hours=4 → 设 5）：worker 空闲满
# (MAX_SESSION_HOURS-1)h 才退出，设 9 会让 4h 的课程白占一整会话。
MAX_SESSION_HOURS = 9
POLL_INTERVAL_SEC = 5  # pull 轮询周期；worker 空闲日志约每 60s 一行（附周期内请求数，见 remote/worker.py）

PLATFORM = "colab" if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ else "kaggle"

print(f"[{time.strftime('%H:%M:%S')}] Mode = {MODE}")
if MODE == "pull":
    print(f"[{time.strftime('%H:%M:%S')}] HUB_URL = '{HUB_URL}'")
    print(f"[{time.strftime('%H:%M:%S')}] HUB_TOKEN len = {len(HUB_TOKEN)}")
else:
    print(f"[{time.strftime('%H:%M:%S')}] PUSH_PORT = {PUSH_PORT}")
    print(f"[{time.strftime('%H:%M:%S')}] PUSH_TOKEN len = {len(PUSH_TOKEN)}")
print(f"[{time.strftime('%H:%M:%S')}] Platform = {PLATFORM}")
print(f"[{time.strftime('%H:%M:%S')}] Max session = {MAX_SESSION_HOURS}h")

---
## 1. 安装依赖

- **GPU 模式**：Kaggle/Colab 默认环境已预装 PyTorch（CUDA 版），仅需确认版本并按需补齐。
- **TPU 模式**（GPU 配额耗尽后接力，独立 20h/周）：走 `torch_xla`，**不要升级/重装 torch**——
  Kaggle TPU 镜像里的 torch 与 torch_xla 是严格配对的一组，装错任一个 TPU 就不可用。


In [ ]:
import os
import subprocess
import sys
import time

import torch


def run(cmd, **kw):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kw)


def _tpu_device_nodes() -> list:
    """本 runtime 映射进来的 TPU 设备节点（只 glob 目录名，**绝不 open**）。

    TPU v5e = /dev/vfio/*（Colab TPU v5e-1 / Kaggle v5e-8）；v3/v4 = /dev/accel*。
    """
    import glob

    return sorted(glob.glob("/dev/vfio/*")) + sorted(glob.glob("/dev/accel*"))


def _tpu_holders() -> list:
    """谁正占着 /dev/vfio/*（pid + cmdline）；扫 /proc/*/fd，同样不 open 设备。

    2026-09-11 Colab 事故收口用：worker 子进程报 busy 时，先看清占用者是
    "本内核自己"（Python 无 release API，只能 Runtime -> Restart）还是
    "别的进程 / 上一轮残留"（kill 即可）。
    """
    import glob

    out = []
    for fdlink in glob.glob("/proc/[0-9]*/fd/*"):
        try:
            tgt = os.readlink(fdlink)
        except OSError:
            continue
        if not tgt.startswith("/dev/vfio"):
            continue
        pid = int(fdlink.split("/")[2])
        try:
            with open(f"/proc/{pid}/cmdline", "rb") as f:
                cmd = f.read().replace(b"\0", b" ").decode("utf-8", "replace").strip()
        except OSError:
            cmd = "?"
        out.append((pid, tgt, cmd))
    return out


# ────────────────────────────── 设备探测：CUDA -> TPU -> CPU ──────────────────────────────
# TPU 硬约束（2026-09-10 接入；2026-09-11 补第 4 条）：
#   1. 必须 torch_xla，且 torch / torch_xla 版本**严格配对**（镜像自带那一对）；
#   2. 必须显式 PJRT_DEVICE=TPU，设备由 torch_xla.device() 取得（不是 torch.device("xla")）；
#   3. **绝对不要 pip install 覆盖 torch** —— 一装即把配对打散，TPU 直接不可用；
#   4. ★★ **本 cell 绝不能 import torch_xla**（2026-09-11 Colab TPU v5e-1 线上事故）：
#      /dev/vfio/* 是**独占** PCI 直通设备（v5e；v3/v4 在 /dev/accel*），同一时刻只能被
#      一个进程持有。本 notebook 真正干活的 worker 跑在 supervise_worker 拉起的**子进程**
#      里（remote/worker.py:852 -> ppo/common.xla_device()）；内核只要碰过设备，子进程就
#      必然报 `TPU initialization failed: open(/dev/vfio/0): Device or resource busy`。
#      ⇒ 探测只查"装没装 / 什么版本"（importlib.util.find_spec + importlib.metadata，
#        两者都不初始化 PJRT 运行时），设备一律留给 worker 子进程去占。
#      权威注释：tools/tpu-probe.py:36-42。
# GPU 配额耗尽后切 TPU 只为**净增 20h/周独立配额**，不替代 GPU。

# ── 多卡开关（Kaggle T4x2 / Colab 单卡）──
# 默认 False = 行为与今天一致（只用 cuda:0）。打开后 >1 卡才包 DataParallel，单卡自动退化。
# ⚠ 为什么默认关：DP 改变梯度归约顺序 -> 末位 ulp 变化 -> 与在跑课程的逐位 A/B 失效。
#   它是**新开一条实验臂**的开关，不是透明加速。
# 实测依据（2026-09-10, Kaggle T4x2, 探针 B_new 段）：单卡 192 ms/step -> 双卡 100 ms/step
#   = 1.92×（接近线性）；反推纯算力 ~184 ms、同步仅 ~8 ms。
USE_MULTI_GPU = False

DEVICE_KIND = "cpu"
DEVICE = "cpu"
_XLA_VER = "未安装"

if torch.cuda.is_available():
    DEVICE_KIND = "cuda"
    _n_gpu = torch.cuda.device_count()
    DEVICE = "cuda-dp" if (USE_MULTI_GPU and _n_gpu > 1) else "cuda"
else:
    # ── TPU 探测：**不 import torch_xla**（硬约束 4）──
    # find_spec 只查模块是否可导入、metadata.version 只读 dist-info，都不初始化 PJRT。
    import importlib.metadata as _md
    import importlib.util as _iu

    if _iu.find_spec("torch_xla") is not None:
        try:
            _XLA_VER = _md.version("torch_xla")
        except Exception:
            _XLA_VER = "（已安装，版本未知）"
        # 只影响**子进程**（supervise_worker 用 dict(os.environ) 继承），本内核不读它。
        os.environ.setdefault("PJRT_DEVICE", "TPU")
        DEVICE_KIND, DEVICE = "tpu", "tpu"
    else:
        print(f"[{time.strftime('%H:%M:%S')}] 无 TPU（torch_xla 未安装）-> CPU")

print(f"[{time.strftime('%H:%M:%S')}] torch {torch.__version__}, device_kind={DEVICE_KIND}")

if DEVICE_KIND == "cuda":
    print(f"[{time.strftime('%H:%M:%S')}]   可见 GPU: {_n_gpu} 张")
    for _i in range(_n_gpu):
        print(f"[{time.strftime('%H:%M:%S')}]     [{_i}] {torch.cuda.get_device_name(_i)}")
    print(f"[{time.strftime('%H:%M:%S')}]   CUDA capability: {torch.cuda.get_device_capability()}")
    if DEVICE == "cuda-dp":
        print(f"[{time.strftime('%H:%M:%S')}]   -> DataParallel 跨 {_n_gpu} 卡"
              "（梯度归约顺序变化，与单卡 run 数值不可逐位比）")
    elif _n_gpu > 1:
        print(f"[{time.strftime('%H:%M:%S')}]   -> 只用第 0 张卡"
              f"（要跨卡把 USE_MULTI_GPU 改 True，或仍用 {_n_gpu} 卡会有收益的 TPU）")
    if int(torch.__version__.split('.')[0]) < 2:
        print(f"[{time.strftime('%H:%M:%S')}] torch 版本过旧，升级中...")
        run(f"{sys.executable} -m pip install --quiet --upgrade torch")
elif DEVICE_KIND == "tpu":
    print(f"[{time.strftime('%H:%M:%S')}]   torch_xla: {_XLA_VER}"
          "（**未导入** —— 设备留给 worker 子进程）")
    print(f"[{time.strftime('%H:%M:%S')}]   TPU 设备节点: "
          f"{_tpu_device_nodes() or '（没看到 /dev/vfio* 或 /dev/accel*）'}")
    _holders = _tpu_holders()
    if _holders:
        for _pid, _tgt, _cmd in _holders:
            _tag = "★本内核★" if _pid == os.getpid() else "其它进程"
            print(f"[{time.strftime('%H:%M:%S')}]   ⚠ {_tgt} 已被占用: "
                  f"pid={_pid} [{_tag}] {_cmd[:90]}")
        print(f"[{time.strftime('%H:%M:%S')}]   ⚠ 占用存在时 worker 子进程必报 busy；"
              "若占用者是本内核，只能 Runtime -> Restart session")
    else:
        print(f"[{time.strftime('%H:%M:%S')}]   vfio 无占用 —— 设备空闲，worker 子进程可正常领取")
    print(f"[{time.strftime('%H:%M:%S')}]   ⚠ 跳过 torch 升级（TPU 需要镜像自带的配对版本）")

print(f"[{time.strftime('%H:%M:%S')}] Dependencies ready -> DEVICE={DEVICE}")


---
## 2. 会话保活

Kaggle GPU 会话最长 9h，每周配额 30h；Colab 免费版约 90min 空闲回收。
通过定期输出（Kaggle）或模拟点击（Colab）防止超时回收。

In [ ]:
import threading

KEEPALIVE_STOP = threading.Event()

if PLATFORM == "colab":
    from IPython.display import Javascript
    from IPython.display import display as ipy_display

    def keepalive_loop():
        """每 60s 点一次 Colab 的 connect 按钮防止超时。"""
        n = 0
        while not KEEPALIVE_STOP.is_set():
            try:
                ipy_display(Javascript("""
                    function clickConnect() {
                        document.querySelector("colab-connect-button")?.click();
                    }
                    setTimeout(clickConnect, 1000);
                """))
                n += 1
            except Exception:
                pass
            KEEPALIVE_STOP.wait(60)
        print(f"[{time.strftime('%H:%M:%S')}] [keepalive] stopped after {n} pings")
else:
    def keepalive_loop():
        """每 120s 打印一次心跳，防止 Kaggle 空闲回收。"""
        n = 0
        while not KEEPALIVE_STOP.is_set():
            print(f"[{time.strftime('%H:%M:%S')}] [keepalive] alive ({n * 2} min elapsed)")
            n += 1
            KEEPALIVE_STOP.wait(120)
        print(f"[{time.strftime('%H:%M:%S')}] [keepalive] stopped after {n} pings ({n * 2} min)")

th = threading.Thread(target=keepalive_loop, daemon=True, name="keepalive")
th.start()
print(f"[{time.strftime('%H:%M:%S')}] Keepalive thread started (every {'60' if PLATFORM == 'colab' else '120'}s, max {MAX_SESSION_HOURS}h)")

---
## 3. 定义 Worker 方法

定义 pull 和 push 两种模式的 GPU worker 方法，供下一步启动时调用。

**Pull 模式**：从 hub 下载 code.zip → 解压到 `sys.path` → 启动 `worker_loop` 轮询领取 job。

**Push 模式**：启动 `remote_worker_serve` HTTP 服务 + cloudflared 隧道暴露到公网，
hub 侧主动推送 job 到此节点。

In [ ]:
def _log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] {msg}", flush=True)


# ────────────────────────────────────────────── Pull 模式 ──

def run_pull_worker() -> int:
    """以 pull 模式连接 hub 轮询 PPO job，返回 worker 子进程最终退出码（0 = 正常）。"""
    import io
    import urllib.error
    import urllib.request
    import zipfile
    from pathlib import Path

    work_dir = Path("/tmp/remote-worker")
    work_dir.mkdir(parents=True, exist_ok=True)

    _log(f"Downloading code.zip from {HUB_URL}/code...")
    req = urllib.request.Request(
        f"{HUB_URL.rstrip('/')}/code",
        headers={"Authorization": f"Bearer {HUB_TOKEN}"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            code_raw = resp.read()
            _log(f"code.zip: {len(code_raw)} bytes (HTTP {resp.status})")
    except urllib.error.HTTPError as e:
        _log(f"FAILED: HTTP {e.code} — {e.read().decode()[:200]}")
        raise SystemExit(1) from None
    except Exception as e:
        _log(f"FAILED: {e}")
        raise SystemExit(1) from None

    code_dir = Path("/tmp/worker-code")
    code_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(code_raw)) as z:
        z.extractall(code_dir)
    sys.path.insert(0, str(code_dir))
    _log(f"code.zip extracted -> {code_dir} (sys.path[0])")

    from remote.worker import supervise_worker

    _log(f"Testing connectivity to hub...")
    req = urllib.request.Request(
        f"{HUB_URL.rstrip('/')}/ping",
        headers={"Authorization": f"Bearer {HUB_TOKEN}"},
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            _log(f"hub HTTP {resp.status}")
    except Exception as e:
        _log(f"CONNECTION FAILED: {e}")
        raise SystemExit(1) from None

    max_idle = max(3600, (MAX_SESSION_HOURS - 1) * 3600)
    _log(f"Starting worker loop (max_idle={max_idle}s, poll_sec={POLL_INTERVAL_SEC}s)...")

    # ── 热替换重启参数（2026-09-11 修复：监督器方案替代 os.execve）──────────────
    # 本地改代码后 hub 会重打 code.zip（sha 变）。旧实现 worker_loop 跑在 kernel
    # 进程内、换码时 os.execve 原地替换 kernel 镜像 → 单元格输出流断（sys.stdout
    # 的 ipykernel 重定向对象丢失）、Jupyter 判定 kernel 死，用户按停止就毁掉整个
    # 云端会话。现在 worker 由 supervise_worker 放到**子进程**：输出逐行转发回
    # 单元格；检测到代码变更时子进程以 HOT_RELOAD_EXIT(=86) 退出，监督器用同一套
    # 参数重新拉起（fresh 进程加载新代码），kernel 与输出流不断。单元格中断时
    # supervise_worker 先终止子进程再上抛，不留孤儿 worker。
    # 这组参数原样传给 supervise_worker（token 走 --token-file，H10：不进进程列表）。
    token_file = work_dir / "hub.token"
    token_file.write_text(HUB_TOKEN, encoding="utf-8")
    try:
        token_file.chmod(0o600)
    except Exception:
        pass
    restart_argv = [
        "--poll", str(HUB_URL),
        "--token-file", str(token_file),
        "--out", str(work_dir),
        "--device", str(DEVICE),
        "--threads", "0",
        "--poll-sec", str(POLL_INTERVAL_SEC),
        "--max-idle-sec", str(max_idle),
    ]

    try:
        rc = supervise_worker(restart_argv)
        return rc
    except KeyboardInterrupt:
        _log(f"Worker interrupted by user")
        return 0


# ────────────────────────────────────────────── Push 模式 ──

def run_push_worker() -> int:
    """以 push 模式启动 worker_server + cloudflared 隧道，等待 hub 推送 job。"""
    import re
    import subprocess as _sp
    from pathlib import Path

    work_dir = Path("/tmp/remote-worker-serve")
    work_dir.mkdir(parents=True, exist_ok=True)

    # ── 查找 / 自动安装 cloudflared ──
    cf_bin = CLOUDFLARED_PATH or _sp.getoutput("where cloudflared 2>nul || which cloudflared 2>/dev/null").strip()
    if not cf_bin:
        _log(f"cloudflared 未找到，尝试自动安装...")
        try:
            cf_install_dir = Path("/usr/local/bin")
            cf_install_dir.mkdir(parents=True, exist_ok=True)
            cf_bin = str(cf_install_dir / "cloudflared")
            _sp.run(
                ["curl", "-fsSL",
                 "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                 "-o", cf_bin],
                check=True, timeout=60,
            )
            os.chmod(cf_bin, 0o755)
            _log(f"cloudflared 已安装到 {cf_bin}")
        except Exception as e:
            _log(f"cloudflared 自动安装失败: {e}")
            cf_bin = ""

    if not cf_bin:
        _log(f"WARNING: cloudflared 不可用——push 模式需要隧道暴露服务")
        _log(f"请手动安装后重试：")
        _log(f"  # Linux: curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
        _log(f"  # macOS: brew install cloudflared")
        _log(f"  # 然后启动: cloudflared tunnel --url http://localhost:{PUSH_PORT}")

    # ── bootstrap 代码（push 模式必需）────────────────────────────────
    # 干净运行时没有仓库代码，remote_worker_serve 不可导入 → 服务进程秒退，
    # 表现为 30s 后的 "server NOT ready"（根因被藏）。先拉一份代码再起服务。
    # ⚠ 语义边界：job 真正执行用的是 hub 随 job 下发的 code.zip（worker_server
    #   收下后入 sys.path、并在**新进程**里跑），bootstrap 代码只用于起服务进程。
    import importlib.util as _ilu

    boot_dir = Path("/tmp/push-bootstrap")
    if _ilu.find_spec("remote_worker_serve") is None:
        url = (
            PUSH_CODE_URL
            or "https://codeload.github.com/HuangJian/battle/tar.gz/refs/heads/main"
        )
        _log(f"remote_worker_serve 不可导入 —— 拉取 bootstrap 代码: {url}")
        try:
            import io as _io
            import tarfile
            import urllib.request as _ureq

            boot_dir.mkdir(parents=True, exist_ok=True)
            with _ureq.urlopen(url, timeout=180) as _r:
                _blob = _r.read()
            with tarfile.open(fileobj=_io.BytesIO(_blob)) as _tf:
                _tf.extractall(boot_dir)
            # tarball 顶层是 <repo>-<branch>/，nn-training 在其下
            _cands = sorted(boot_dir.glob("*/nn-training"))
            if _cands:
                boot_dir = _cands[0]
            sys.path.insert(0, str(boot_dir))
            _log(f"bootstrap 代码就位 -> {boot_dir}")
        except Exception as e:
            _log(f"bootstrap 代码拉取失败: {e}")
            _log("push 模式无法启动：请改用 pull 模式，或手动把 nn-training/ 放到运行时并设 PUSH_CODE_URL")
            return -1
    else:
        _log("remote_worker_serve 已可导入 —— 跳过 bootstrap")

    # ── 启动 worker_server ──
    _log(f"Starting worker server on 0.0.0.0:{PUSH_PORT}...")
    serve_log = work_dir / "serve.log"
    with open(serve_log, "w") as log_f:
        _serve_env = dict(os.environ)
        _pp = str(boot_dir) + os.pathsep + _serve_env.get("PYTHONPATH", "")
        _serve_env["PYTHONPATH"] = _pp.rstrip(os.pathsep)
        serve_proc = _sp.Popen(
            [sys.executable, "-u", "-m", "remote_worker_serve",
             "--port", str(PUSH_PORT),
             "--token", PUSH_TOKEN,
             "--work", str(work_dir),
             "--device", DEVICE],
            stdout=log_f, stderr=_sp.STDOUT, env=_serve_env,
        )
    _log(f"Worker server started (PID {serve_proc.pid})")

    # 等待服务就绪
    import urllib.request as _ur

    def _ping_ok() -> bool:
        try:
            req = _ur.Request(
                f"http://127.0.0.1:{PUSH_PORT}/ping",
                headers={"Authorization": f"Bearer {PUSH_TOKEN}"},
            )
            with _ur.urlopen(req, timeout=5) as r:
                return r.status == 200
        except Exception:
            return False

    t0 = time.time()
    while time.time() - t0 < 30:
        if _ping_ok():
            break
        time.sleep(1)
    if _ping_ok():
        _log(f"Worker server ready")
    else:
        _log(f"Worker server NOT ready (30s timeout) — check {serve_log}")
        serve_proc.kill()
        return -1

    # ── 启动 cloudflared tunnel ──
    cf_url = None
    cf_proc = None
    if cf_bin:
        _log(f"Starting cloudflared tunnel...")
        cf_log = work_dir / "cloudflared.log"
        with open(cf_log, "w") as log_f:
            cf_proc = _sp.Popen(
                [cf_bin, "tunnel", "--url", f"http://localhost:{PUSH_PORT}", "--logfile", str(cf_log)],
                stdout=log_f, stderr=_sp.STDOUT,
            )

        t0 = time.time()
        while time.time() - t0 < 60:
            try:
                text = cf_log.read_text(encoding="utf-8", errors="replace")
                urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
                if urls:
                    cf_url = urls[-1]
                    break
            except Exception:
                pass
            if cf_proc.poll() is not None:
                _log(f"cloudflared exited early (code {cf_proc.returncode})")
                break
            time.sleep(2)

        if cf_url:
            _log(f"cloudflared tunnel URL: {cf_url}")
            _log(f"将此 URL 配置到 hub 侧 rl-config.json 的 nodes 条目")
        else:
            _log(f"cloudflared tunnel URL not obtained (timeout/error)")
    else:
        _log(f"cloudflared 不可用，跳过隧道启动")

    # ── 等待 job（保持会话活跃） ──
    _log(f"Waiting for jobs... (keepalive active)")
    _log(f"hub 侧配置 push 节点后，job 会自动推送至此")

    t_start = time.time()
    try:
        while True:
            if serve_proc.poll() is not None:
                _log(f"Worker server exited (code {serve_proc.returncode})")
                break
            if time.time() - t_start > MAX_SESSION_HOURS * 3600:
                _log(f"Max session reached ({MAX_SESSION_HOURS}h)")
                break
            time.sleep(30)
    except KeyboardInterrupt:
        _log(f"Interrupted by user")
    finally:
        if cf_proc and cf_proc.poll() is None:
            cf_proc.kill()
            _log(f"cloudflared stopped")
        if serve_proc.poll() is None:
            serve_proc.kill()
            _log(f"Worker server stopped")

    return 0

---
## 4. 启动 GPU Worker

根据 `MODE` 选择启动 pull 或 push 模式的 GPU worker。
可以随时中断此单元格（`Kernel → Interrupt` 或 `Runtime → Interrupt execution`），worker 会优雅退出。
中断后如还有配额，可重新运行此单元格继续工作。

In [ ]:
t_start = time.time()

if MODE == "pull":
    print(f"\n{'='*60}")
    print(f"  [battle-rl] 工作模式: PULL")
    print(f"  [battle-rl] 连接 hub: {HUB_URL}")
    print(f"  [battle-rl] 平台: {PLATFORM}")
    print(f"  [battle-rl] 设备: {DEVICE}")
    print(f"  [battle-rl] 最大会话: {MAX_SESSION_HOURS}h")
    print(f"  [battle-rl] 行为: 轮询 hub 领取 PPO job → 执行 → 回传结果")
    print(f"{'='*60}\n")

    n = run_pull_worker()

    elapsed = time.time() - t_start
    print(f"\n[{time.strftime('%H:%M:%S')}] [battle-rl] Worker exited: rc={n}")
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] Session duration: {elapsed/60:.1f} min")

elif MODE == "push":
    print(f"\n{'='*60}")
    print(f"  [battle-rl] 工作模式: PUSH")
    print(f"  [battle-rl] 监听端口: {PUSH_PORT}")
    print(f"  [battle-rl] 平台: {PLATFORM}")
    print(f"  [battle-rl] 设备: {DEVICE}")
    print(f"  [battle-rl] 最大会话: {MAX_SESSION_HOURS}h")
    print(f"  [battle-rl] 行为: 启动 worker_server + cloudflared → 等待 hub 推送 job")
    print(f"{'='*60}\n")

    ret = run_push_worker()

    elapsed = time.time() - t_start
    print(f"\n[{time.strftime('%H:%M:%S')}] [battle-rl] Session duration: {elapsed/60:.1f} min")

else:
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] 未知 MODE={MODE!r}，可选 'pull' 或 'push'")

# 停止保活
KEEPALIVE_STOP.set()

---
## 5. 停止保活（备用）

如果提前中断了步骤 4，运行此单元格停止保活线程。
正常退出时保活会自动停止，无需手动运行此单元格。

In [ ]:
KEEPALIVE_STOP.set()
print(f"[{time.strftime('%H:%M:%S')}] Keepalive stopped")

---
## 6. 会话管理与故障排查

### GPU 配额管理

| 平台 | 限制 | 说明 |
|---|---|---|
| Kaggle | 每周 30h GPU，单次 9h | 每周一 UTC 重置，worker 空闲超时自动退出 |
| Colab 免费 | ~90min 空闲回收 | 保活线程每 60s 模拟点击 |
| Colab Pro | 更长的使用时间 | 同上，保活线程同样适用 |

### 中断后重连

- 中断后重新运行步骤 4 即可继续
- hub 的 job 幂等机制保证不会重复训练同一轮

### 预期日志

**hub 端**（pull 模式）：
```
[hub-server] "GET /jobs/next HTTP/1.1" 200 -
[hub-server] "GET /jobs/{id}/payload HTTP/1.1" 200 -
[hub-server] "POST /jobs/{id}/result HTTP/1.1" 200 -
```

**worker 端**（pull 模式）：
```
[worker] job {id} claimed — downloading payload
[worker] job {id}: code.zip unpacked (N bytes, M .py files) -> sys.path[0]
[worker] job {id}: PPO done in {sec}s, steps={n} chunks={m} kl={k}
[worker] job {id} done — result accepted
```

**hub 端**（push 模式）：
```
[push] job {id} 已推送到 {node_url}
[push] wait_result: job {id} done
[hub] weights landed -> {out_weights}
```

**worker 端**（push 模式）：
```
[worker-serve] job {id} accepted
[worker-serve] job {id}: PPO done in {sec}s
[worker-serve] job {id} done — result ready for pickup
```

### 常见问题

| 症状 | 原因 | 处理 |
|---|---|---|
| `CONNECTION FAILED` | hub tunnel 未启动或 URL 过期 | 确认 hub 端 tunnel 运行中，更新 HUB_URL |
| `HTTP 401` | token 不匹配 | 检查凭证与 hub 端 rl-config.json 一致 |
| `payload_sha256 不匹配` | 传输损坏 | 自动重试，hub 的 job 幂等机制保证不重复 |
| 长时间无 job | rollout 采集未完成 | hub 端 rollout 采集完成后自动发布 job |
| Kaggle 配额耗尽 | 30h/周 GPU 用完 | 等待周一重置，或使用 Colab Pro/AutoDL 替代 |
| push 模式 cloudflared 启动失败 | 镜像缺少 cloudflared | 手动安装，或改用 pull 模式 |
| `TPU initialization failed: open(/dev/vfio/0): Device or resource busy` | TPU 是**独占**的 PCI 直通设备（`/dev/vfio/*`），已被别的进程持有（常见：本内核 import 过 torch_xla） | 本内核占用只能 `Runtime → Restart session`（Python 层无 release API）；先看**步骤 1** 输出的「vfio 无占用 / 已被占用」两行定位占用者 |

---
## 附录：与 hub-start.ts 的配合

### Pull 模式

1. 本地起基建 + 训练：`bun run train`（训练控制台；selfNode → hubServer →
   cloudflared → trainingLoop 四步）。命令行等价形式：
   `bun tools/training/train.ts --script run_rl.py -- --course <course> --ppo remote`
   （`tools/hub-start.ts` 已拆分进 `tools/training/hub.ts`，旧命令不再存在）
2. 取连接信息：隧道 URL 由 cloudflared 步骤写回 `nn-training/rl-config.json`
   的 `rl.remote_hub_url`，token 在同文件的 `rl.remote_token`
   （旧版的“📋 Kaggle 粘贴用”打印已取消）
3. 在本 notebook 的 `⚙️ 参数配置` 中填入
4. 设置 `MODE = "pull"`
5. 依次运行各单元格

### Push 模式

1. 在本 notebook 设置 `MODE = "push"`，填入 PUSH_TOKEN
2. 运行 notebook，步骤 4 启动后等待 cloudflared 打印隧道 URL
3. 在 hub 侧 `rl-config.json` 的 nodes 中添加：
   ```json
   {
     "id": "gpu-worker",
     "url": "<cloudflared-tunnel-url>",
     "authKey": "<PUSH_TOKEN>",
     "concurrency": 1,
     "enabled": true,
     "gpu_push": true
   }
   ```
4. 本地起基建：`bun run train`（或 `bun tools/training/train.ts --script
   run_rl.py -- --course <course> --ppo remote`），hub 会自动把 job 推到 GPU 节点

### 可用课程

以 `nn-training/curricula/` 目录为准（`bun tools/training/train.ts --script
run_rl.py -- --course <不存在>` 会打印全部候选）。常见课程：

- `p1-onset` - 单敌近战
- `p4-onset` - 四面围攻（4 敌混编）
- `p4-horizon` - 水平进攻
- `c5-margin` / `c6-margin` / `c6b-margin` - margin 系列（c6b = c6 判否后的
  重开腿：`seed_rotate` 600 局/轮、`wTick` 0.001、`stage_clear` 6.0）
- `s1` ~ `s5` - 专项技能课程
- `s-dodge` - 闪避训练
- `s3-balanced` - 平衡型课程

### c6b-margin 专项注意

| 项 | 值 | 对 worker 的影响 |
|---|---|---|
| `max_hours` | 4 | 把 `MAX_SESSION_HOURS` 调到 5，否则 worker 空转到 8h 才退 |
| `seed_rotate` | 600 | 单轮样本 4×（600 局 ≈ 26k samples，obs 约 440 KB/局未压缩）；
| | | job 目录不做历史清理，20 轮累计约 5 GB —— 关注运行时磁盘 |
| job 超时 | 1800s（hub 硬编码） | 实测 150 局 ≈ 70s，600 局量级远低于超时，安全 |
| `ent_break` 0.25 | hub 侧 | 远端回传 entropy，熔断仍在 hub 生效，worker 无需感知 |
| `gates` / `eval_every` 3 | hub 侧 | 与 worker 无关 |